# BeRL — Comprehensive, eval-grounded re-assessment of all Completed runs

**Goal:** find a winning *training recipe* by ranking every Completed phase-stability run on
**authoritative eval accuracy** — the full `subsample300` `val/test_score/<bench>` numbers —
instead of training reward / rollout response-length, which are **misleading**.

### Why not reward / resp_len (the PS129 lesson)
PS129 (Gemma-2, power-k7, actor-RM) had the single biggest HM jump (0.085→0.43) but:
its training rollout `resp_len` **collapsed to ~1.5 tokens**, `kl` blew up to the ~10 cap, and
reward inflated to 15.7 (actor-as-RM self-inflation). Yet its *eval* CoTs stayed coherent and
gsm8k/mmlu **rose** — so the eval gain was real but the run is training-unstable / reward-hacks
the behavior objective. **Conclusion: rank on eval accuracy, gate on eval-CoT quality, and treat
reward/KL/resp_len as stability *annotations*, not the score.**

### Metric definitions
* **ToM HM** = harmonic mean over the 24 ToM benchmarks (excludes gsm8k, mmlu), avg-then-HM over
  the last-N eval iters (identical convention to `scripts/score_run.py`).
* **dHM** = `HM(last3) - HM(step0)` — vs each run's OWN step-0 baseline, ranked **within model
  family** (Qwen2.5 base HM≈0.42, Qwen3≈0.34/0.16, Gemma-2≈0.09 are not comparable in absolute).
* **stability** = std of per-iter ToM HM over the last 5 evals (prefer a sustained plateau).
* **regression** = dgsm8k, dmmlu (reported separately, never folded into HM).
* **eval-quality gate** = format-parseable & correct fraction from `[val sample|step|score]` blocks,
  plus eval-CoT char length (catches 'empty CoT wins binary MC by luck', which test_score alone can't).
* **health annotation** = final KL (flag near the ~10 cap), min rollout resp_len (collapse).


## 0. Setup
Run from the repo root in the `tom` conda env. The heavy lifting lives in
`scripts/reassess_runs.py` (rg-accelerated extraction, cached to `analysis/cache/`).


In [1]:
import sys, os, importlib
# locate repo root (dir containing scripts/reassess_runs.py) so the notebook runs from anywhere
root = os.getcwd()
while root != '/' and not os.path.exists(os.path.join(root,'scripts','reassess_runs.py')):
    root = os.path.dirname(root)
os.chdir(root); sys.path.insert(0, os.path.join(root,'scripts'))
print('repo root:', root)
import reassess_runs as R
importlib.reload(R)
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

repo root: /mnt/home/judekhouja/repo/BeRL


## 1. Extract + score every Completed run
Resolves each Completed tracker row to its disk log (picking the retry attempt with the most eval
iters), extracts eval trajectories + health + val-sample quality, and computes the metrics.
First run parses ~72 GB (a few minutes); results are cached so reruns are instant.


In [2]:
rows = R.assess_all(status='completed')
df = pd.DataFrame(rows)
df = df[df['n_iters'].fillna(0) >= 2].copy()
print(len(df), 'scored runs')
df.to_csv('analysis/reassess.csv', index=False)


[1/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp0_ec0.0_lp-dcfg_smoke_mix-Q
[2/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp0_ec0.001_lp-dcfg_smoke_mix
[3/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp5_ec0.0_lp-dcfg_smoke_mix-Q
[4/152] Phase-stability-Pm1swp_rmf_kl0.01_lr5e-7_fp5_ec0.001_lp-dcfg_smoke_mix
[5/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp0_ec0.0_lp-dcfg_smoke_mix-Q
[6/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp0_ec0.001_lp-dcfg_smoke_mix
[7/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp5_ec0.0_lp-dcfg_smoke_mix-Q
[8/152] Phase-stability-Pm1swp_rmf_kl0.01_lr1e-6_fp5_ec0.001_lp-dcfg_smoke_mix
[9/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.0_lp-dcfg_smoke_mix-Q
[10/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.001_lp-dcfg_smoke_mix
[11/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_ec0.0_lp-dcfg_smoke_mix-Q
[12/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_ec0.001_lp-dcfg_smoke_mix
[13/152] Phase-stability-Pm1swp_rmf_kl0.05_lr1e-6

[81/152] Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_ec0.0_pk5-dcfg_smoke_mix-
[82/152] Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_ec0.001_pk5-dcfg_smoke_mi
[83/152] Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp0_ec0.001_pk5-dcfg_smoke_mi
[84/152] Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_ec0.0_pk5-dcfg_smoke_mix-
[85/152] Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_ec0.001_pk5-dcfg_smoke_mi
[86/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr1e-6_fp0_ec0.0_lp-dcfg_smoke_
[87/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr1e-6_fp0_ec0.001_lp-dcfg_smok
[88/152] Phase-stability-Pm1w2gemma2_rma_kl0.05_lr1e-6_fp0_ec0.0_lp-dcfg_smoke_
[89/152] Phase-stability-Pm1w2gemma2_rma_kl0.05_lr1e-6_fp5_ec0.001_lp-dcfg_smok
[90/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp0_ec0.0_pk5_llm6-dcfg_
[91/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp5_ec0.0_pk5_llm6-dcfg_
[92/152] Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_fp5_ec0.001_pk5_llm6-dcf
[93/152] Phase-stability-Pm1w2gemma2_rmf

152 scored runs


[145/152] Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_fp0_ec0.0_pk7_llm6-dcfg_s
[146/152] Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_fp0_ec0.001_pk7_llm6-dcfg
[147/152] Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_fp5_ec0.0_pk7_llm6-dcfg_s
[148/152] Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_fp5_ec0.0_pk7_llm4-dcfg_s
[149/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.001_pk5_llm4-dcfg_smo
[150/152] Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp0_ec0.001_pk5_llm4-dcfg_smo
[151/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.001_pk7_llm6-dcfg_smo
[152/152] Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_ec0.001_pk7_llm4-dcfg_smo


## 2. Health gate (clean vs reward-hacking-unstable)
A run is **clean** if it neither blew up KL (`kl_final < 1.0`) nor collapsed rollouts
(`resp_len_min >= 30`). This is an annotation used to separate *adoptable* recipes from unstable
outliers like PS129 — it does **not** remove them from the eval-accuracy ranking.


In [3]:
for c in ['d_hm','d_avg','kl_final','resp_len_min','d_gsm8k','d_mmlu','hm_step0','hm_last3','hm_late_std']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['clean'] = (df['kl_final'] < 1.0) & (df['resp_len_min'] >= 30)
print('frozen RM  KL-blowups:', ((df.rm_mode=='frozen') & (df.kl_final>1)).sum(), '/', (df.rm_mode=='frozen').sum())
print('actor  RM  KL-blowups:', ((df.rm_mode=='actor')  & (df.kl_final>1)).sum(), '/', (df.rm_mode=='actor').sum())


frozen RM  KL-blowups: 0 / 80
actor  RM  KL-blowups: 12 / 72


## 3. Per-family leaderboard (ranked by dHM)
Absolute HM is dominated by base model, so we rank **within** family.


In [4]:
cols = ['run','reward','power_k','ll_min','rm_mode','kl','lr','fp','ec',
        'd_hm','d_avg','hm_step0','hm_last3','hm_late_std','eval_correct','d_gsm8k','d_mmlu',
        'kl_final','resp_len_min','clean','n_iters']
for fam in ['Qwen2.5','Qwen3','Gemma-2']:
    print(f'\n===== {fam} — top 10 by dHM =====')
    sub = df[df.model==fam].sort_values('d_hm', ascending=False)[cols].head(10)
    display(sub)



===== Qwen2.5 — top 10 by dHM =====


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,eval_correct,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
150,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_e...,power,7.0,6.0,frozen,0.05,5e-7,0.0,0.001,0.0547,0.0236,0.4149,0.4696,0.0016,0.513,0.0043,0.1413,0.084,50.2,True,14
82,Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp0_e...,power,5.0,6.0,actor,0.05,1e-6,0.0,0.001,0.0541,0.0235,0.4158,0.4699,0.0060,0.474,-0.0433,0.1480,0.063,43.4,True,19
80,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.000,0.0529,0.0202,0.4158,0.4687,0.0018,0.474,-0.0090,0.1613,0.093,39.6,True,20
84,Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_e...,power,5.0,6.0,actor,0.05,1e-6,5.0,0.001,0.0526,0.0214,0.4166,0.4693,0.0033,0.526,-0.0323,0.1617,0.089,40.6,True,20
83,Phase-stability-Pm1swp_rma_kl0.05_lr1e-6_fp5_e...,power,5.0,6.0,actor,0.05,1e-6,5.0,0.000,0.0520,0.0207,0.4150,0.4670,0.0032,0.551,0.0057,0.1633,0.098,46.3,True,20
70,Phase-stability-Pm1swp_rmf_kl0.05_lr1e-6_fp5_e...,power,5.0,6.0,frozen,0.05,1e-6,5.0,0.000,0.0501,0.0222,0.4178,0.4679,0.0049,0.564,-0.0990,0.1367,0.087,44.6,True,20
81,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.001,0.0491,0.0214,0.4159,0.4650,0.0055,0.615,-0.0090,0.1453,0.084,43.9,True,20
79,Phase-stability-Pm1swp_rma_kl0.05_lr5e-7_fp0_e...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0483,0.0197,0.4154,0.4637,0.0025,0.526,-0.0323,0.1443,0.100,42.0,True,20
67,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp5_e...,power,5.0,6.0,frozen,0.05,5e-7,5.0,0.001,0.0478,0.0217,0.4208,0.4686,0.0012,0.500,-0.0083,0.1670,0.092,48.8,True,19
77,Phase-stability-Pm1swp_rma_kl0.01_lr1e-6_fp5_e...,power,5.0,6.0,actor,0.01,1e-6,5.0,0.001,0.0478,0.0161,0.4156,0.4633,0.0062,0.487,-0.0900,0.1373,0.160,40.5,True,20



===== Qwen3 — top 10 by dHM =====


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,eval_correct,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
115,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,frozen,0.05,1e-6,0.0,0.001,0.0632,0.0397,0.3415,0.4046,0.0089,0.462,0.0173,0.0607,0.011,608.9,True,14
114,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,frozen,0.05,1e-6,0.0,0.000,0.0585,0.0377,0.3403,0.3988,0.0053,0.474,0.0043,0.0490,0.014,622.6,True,14
142,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,4.0,actor,0.05,5e-7,5.0,0.000,0.0339,0.0256,0.3342,0.3681,0.0182,0.423,0.0180,0.0370,0.002,566.6,True,39
139,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.001,0.0185,0.0076,0.3422,0.3607,0.0076,0.423,0.0177,0.0117,0.002,592.9,True,39
124,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,5.0,6.0,frozen,0.05,5e-7,5.0,0.001,0.0103,0.0044,0.3423,0.3526,0.0115,0.449,0.0120,0.0040,0.003,567.2,True,39
131,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,7.0,6.0,frozen,0.05,5e-7,5.0,0.000,0.0093,0.0050,0.3419,0.3512,0.0141,0.513,0.0007,-0.0077,0.004,586.0,True,39
137,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0069,-0.0021,0.1605,0.1674,0.0038,0.256,-0.0157,0.0297,0.003,439.1,True,20
129,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr5e-7_f...,power,7.0,6.0,frozen,0.05,5e-7,0.0,0.000,0.0068,0.0036,0.1605,0.1673,0.0063,0.359,-0.0003,0.0447,0.003,438.2,True,39
145,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,7.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0048,0.0044,0.1607,0.1655,0.0140,0.346,0.0207,0.0257,0.003,430.3,True,39
138,Phase-stability-Pm1w2qwen3_rma_kl0.05_lr5e-7_f...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.000,-0.0011,-0.0029,0.1605,0.1594,0.0060,0.231,-0.0040,0.0117,0.003,432.0,True,39



===== Gemma-2 — top 10 by dHM =====


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,eval_correct,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
109,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,6.0,actor,0.05,5e-7,0.0,0.000,0.3445,0.1919,0.0853,0.4298,0.0068,0.513,0.1530,0.1163,9.852,1.5,False,20
92,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,0.05,5e-7,0.0,0.000,0.1326,0.1041,0.0930,0.2256,0.0253,0.397,0.0897,0.0623,0.095,64.6,True,39
108,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,4.0,actor,0.05,5e-7,5.0,0.001,0.1159,0.0807,0.0929,0.2088,0.0203,0.385,0.0963,0.0467,0.029,61.9,True,39
105,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,6.0,actor,0.05,5e-7,5.0,0.001,0.1014,0.0998,0.0930,0.1944,0.0439,0.423,0.2507,0.0907,0.580,10.3,False,38
103,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.000,0.0886,0.0643,0.0854,0.1740,0.0510,0.397,0.1947,0.0750,1.845,21.4,False,20
104,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,5.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0818,0.0885,0.0930,0.1748,0.0273,0.308,0.2067,0.0497,3.392,19.4,False,39
113,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,4.0,actor,0.05,5e-7,5.0,0.000,0.0795,0.0756,0.0930,0.1725,0.0390,0.449,0.0553,0.0577,0.042,65.0,True,39
95,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,0.05,5e-7,5.0,0.001,0.0751,0.0968,0.0930,0.1681,0.0246,0.513,0.0563,0.0257,0.077,84.0,True,39
110,Phase-stability-Pm1w2gemma2_rma_kl0.05_lr5e-7_...,power,7.0,6.0,actor,0.05,5e-7,0.0,0.001,0.0619,0.0485,0.0930,0.1549,0.0222,0.359,0.2163,0.0787,0.553,27.6,False,39
100,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,7.0,4.0,frozen,0.05,5e-7,0.0,0.001,0.0552,0.0859,0.0930,0.1482,0.0426,0.449,0.0830,0.0367,0.059,69.1,True,20


## 4. Clean winner per family (health-gated)
The best *adoptable* recipe per family = highest dHM among `clean` runs.


In [5]:
winners = (df[df.clean].sort_values('d_hm', ascending=False)
             .groupby('model', as_index=False).first())
display(winners[cols])


,run,reward,power_k,ll_min,rm_mode,kl,lr,fp,ec,d_hm,d_avg,hm_step0,hm_last3,hm_late_std,eval_correct,d_gsm8k,d_mmlu,kl_final,resp_len_min,clean,n_iters
0,Phase-stability-Pm1w2gemma2_rmf_kl0.05_lr5e-7_...,power,5.0,4.0,frozen,0.05,5e-7,0.0,0.000,0.1326,0.1041,0.0930,0.2256,0.0253,0.397,0.0897,0.0623,0.095,64.6,True,39
1,Phase-stability-Pm1swp_rmf_kl0.05_lr5e-7_fp0_e...,power,7.0,6.0,frozen,0.05,5e-7,0.0,0.001,0.0547,0.0236,0.4149,0.4696,0.0016,0.513,0.0043,0.1413,0.084,50.2,True,14
2,Phase-stability-Pm1w2qwen3_rmf_kl0.05_lr1e-6_f...,log_prob,2.0,8.0,frozen,0.05,1e-6,0.0,0.001,0.0632,0.0397,0.3415,0.4046,0.0089,0.462,0.0173,0.0607,0.011,608.9,True,14


## 5. Winning archetype (reward x RM), clean only
Which knob combinations robustly help vs hurt.


In [6]:
arch = (df[df.clean].groupby(['model','reward','rm_mode'])['d_hm']
          .agg(['count','mean','max']).reset_index()
          .sort_values('mean', ascending=False))
display(arch[arch['count']>=2])


,model,reward,rm_mode,count,mean,max
2,Gemma-2,power,actor,4,0.041400,0.1159
6,Qwen2.5,power,actor,25,0.031872,0.0541
9,Qwen3,log_prob,frozen,4,0.007300,0.0632
3,Gemma-2,power,frozen,11,0.003509,0.1326
7,Qwen2.5,power,frozen,30,0.002867,0.0547
10,Qwen3,power,actor,12,-0.012167,0.0339
0,Gemma-2,log_prob,actor,2,-0.014500,-0.0026
11,Qwen3,power,frozen,14,-0.021043,0.0103
4,Qwen2.5,log_prob,actor,11,-0.036473,0.0268
5,Qwen2.5,log_prob,frozen,16,-0.082825,0.0163


## 6. Eval-CoT quality gate (anti-MC-luck)
Confirm the winners emit real, non-empty eval CoTs (not short outputs winning binary MC by chance).
Extracts the actual generation after the assistant turn marker from `[val sample]` blocks.


In [7]:
import re
HDR = re.compile(r'\[val sample \| step=(\d+) \| source=(\w+) \| score=(-?[0-9.]+)\]')
def eval_gen_lengths(path):
    lines=[re.sub(r'^\(main_task pid=\d+\)\s?','',l) for l in open(path,encoding='utf-8',errors='ignore')]
    n=len(lines); i=0; per={}
    while i<n:
        m=HDR.search(lines[i])
        if m:
            step=int(m.group(1)); j=i+1; gen=None
            while j<n and not HDR.search(lines[j]):
                s=lines[j].strip()
                if s.endswith('<start_of_turn>model') or s.endswith('assistant'):
                    k=j+1
                    while k<n and lines[k].strip()=='': k+=1
                    gen=re.sub(r'^\(main_task pid=\d+\)\s?','',lines[k]).strip() if k<n else ''
                    break
                j+=1
            per.setdefault(step,[]).append(len(gen) if gen else 0)
            i=j
        else: i+=1
    return per
for _,w in winners.iterrows():
    per=eval_gen_lengths(w['log']); steps=sorted(per)
    if not steps: continue
    last=[x for s in steps[-3:] for x in per[s]]
    print(f"{w['model']:8} eval-CoT chars: step0={np.mean(per[steps[0]]):.0f} last3={np.mean(last):.0f} "
          f"empty(<5)={sum(1 for x in last if x<5)}/{len(last)}  | {w['reward']} k{w['power_k']} {w['rm_mode']}RM")


Gemma-2  eval-CoT chars: step0=144 last3=408 empty(<5)=0/78  | power k5.0 frozenRM


Qwen2.5  eval-CoT chars: step0=452 last3=445 empty(<5)=0/78  | power k7.0 frozenRM


Qwen3    eval-CoT chars: step0=39 last3=39 empty(<5)=0/78  | log_prob k2.0 frozenRM


## 7. Verdict
* **Winning recipe is family-specific** (do not assume cross-family transfer):
  * **Qwen2.5** — power reward, k=7, ll_min=−6, **frozen RM**, kl=0.05, lr=5e-7, fp=0, ec=0.001 →
    dHM ≈ +0.055 (0.415→0.470), gsm8k flat. Small because the base is already strong.
  * **Qwen3** — **log_prob** reward (k=2, ll_min=−8), **frozen RM**, kl=0.05, lr=1e-6, fp=0 →
    dHM ≈ +0.063 (0.342→0.405), gsm8k +0.017. (power barely moves Qwen3.)
  * **Gemma-2** — power reward, k=5, ll_min=−4, **frozen RM**, kl=0.05, lr=5e-7, fp=0, ec=0.0 →
    dHM ≈ +0.133 (0.093→0.226), gsm8k +0.090. **Largest genuine, stable gain.**
* **Cross-family backbone of every clean winner: frozen RM · kl=0.05 · fp=0 · lr∈{5e-7,1e-6}.**
* **actor-RM is the reward-hacking destabiliser:** 12/72 actor runs blew up KL vs **0/80** frozen.
  PS129 (Gemma actor-k7) has the biggest raw dHM (+0.344) but reward-hacks (kl 9.85, rollout 1.5 tok)
  — real eval gain, **not an adoptable recipe**.
